<a href="https://colab.research.google.com/github/MohammedAL-Ani/Stuttgart/blob/main/Stuttgart%20Am%20Neckartor/beijing/Beijing_Nongzhanguan_NO2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import subprocess, sys
try:
    import vmdpy
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vmdpy"], check=True)

import os
STATION    = "Nongzhanguan"
FILE_PATH  = f"/content/PRSA_Data_{STATION}_20130301-20170228.csv"
POLLUTANT  = "NO2"
DATE_COL   = "Date / Time"
VALUE_COL  = "Value"
DATE_START = "2013-03-01 00:00"
DATE_END   = "2017-02-28 23:00"
OUT_DIR    = f"/content/beijing_{STATION}_{POLLUTANT}"
USE_DRIVE  = False
SKIP_DONE  = True
AUTO_DOWNLOAD = True

TRACKS     = ["RAW", "STL", "STL-VMD", "STL-ARIMA"]
HORIZONS   = [1, 24]
LOOKBACK   = 24
TRAIN_FRAC = 0.70
VAL_FRAC   = 0.10
MAX_GAP_INTERP = 6



DECOMP_WINDOW = 336
STL_PARAMS = dict(period=24, seasonal=7, trend=47, low_pass=25,
                  seasonal_deg=1, trend_deg=1, low_pass_deg=1, robust=True)
STL_FIT    = dict(inner_iter=2, outer_iter=15)


VMD_PARAMS  = dict(alpha=2000, tau=0.0, DC=0, init=1, tol=1e-7)
VMD_K       = 8
VMD_K_RANGE = list(range(2, 13))
VMD_K_SAMPLE_WINDOWS = 200
VMD_CF_GAP_MIN = 0.01


ARIMA_PQ_MAX     = 3
RAW_ARIMA_PQ_MAX = 2
ARIMA_CRITERION  = "bic"


XGB_TRIALS     = 25
XGB_MAX_TREES  = 1500
DL_MAX_EPOCHS  = 100
DL_PATIENCE    = 10
BATCH_SIZE     = 256
TUNE_DL        = True
SEED           = 42
N_JOBS         = -1
PEAK_PCT       = 90
RELRES_MIN     = 1.0


QUICK = os.environ.get("PM_QUICK", "0") == "1"
if QUICK:
    FILE_PATH = os.environ.get("PM_FILE", FILE_PATH)
    OUT_DIR = os.environ.get("PM_OUT", OUT_DIR)
    DECOMP_WINDOW, XGB_TRIALS, DL_MAX_EPOCHS, DL_PATIENCE = 168, 3, 4, 2
    VMD_K_RANGE, VMD_K_SAMPLE_WINDOWS, TUNE_DL = [2, 3, 4], 20, False
    VMD_K, USE_DRIVE = int(os.environ.get("PM_K", "4")), False
    HORIZONS = [int(h) for h in os.environ.get("PM_H", "1,24").split(",")]

assert DECOMP_WINDOW % 2 == 0, "DECOMP_WINDOW must be even (vmdpy drops the last sample for odd lengths)"
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        OUT_DIR = f"/content/drive/MyDrive/beijing_{STATION}_{POLLUTANT}"
    except Exception as e:
        print("Google Drive not available, saving locally:", e)
os.makedirs(OUT_DIR, exist_ok=True)

POLL_TAG   = f"{STATION}_" + POLLUTANT.replace(".", "")
UPLOAD_DIR = "/content" if os.path.isdir("/content") else os.path.dirname(os.path.abspath(OUT_DIR))
print("Station:", STATION, "| Pollutant:", POLLUTANT, "| file:", FILE_PATH, "| results folder:", OUT_DIR)


Station: Nongzhanguan | Pollutant: NO2 | file: /content/PRSA_Data_Nongzhanguan_20130301-20170228.csv | results folder: /content/beijing_Nongzhanguan_NO2


In [ ]:

import glob, zipfile
cands = sorted(glob.glob(f"{UPLOAD_DIR}/{POLL_TAG}_checkpoint*.zip") + glob.glob(f"{UPLOAD_DIR}/{POLL_TAG}_FINAL_results*.zip"),
               key=os.path.getmtime)
if cands:
    with zipfile.ZipFile(cands[-1]) as z:
        z.extractall(OUT_DIR)
    print("Restored checkpoint:", cands[-1])
    print("Contents:", sorted(os.listdir(OUT_DIR)))
else:
    print("No checkpoint zip found in", UPLOAD_DIR, "- starting fresh (files already in", OUT_DIR, "are still reused).")


No checkpoint zip found in /content - starting fresh (files already in /content/beijing_Nongzhanguan_NO2 are still reused).


In [ ]:

import time, random, json, platform, warnings
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg") if QUICK else None
import matplotlib.pyplot as plt
import torch, torch.nn as nn
import statsmodels, sklearn, xgboost, scipy
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.stats.diagnostic import acorr_ljungbox, het_breuschpagan, het_arch
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from scipy import stats
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor
from vmdpy import VMD
from joblib import Parallel, delayed
from importlib.metadata import version as _pkg_version

warnings.filterwarnings("ignore")
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
XGB_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

REPORT = {}


REPORT["Software_versions"] = pd.DataFrame([
    ("Python", platform.python_version()), ("numpy", np.__version__), ("pandas", pd.__version__),
    ("statsmodels (STL, ARIMA, tests)", statsmodels.__version__),
    ("vmdpy (VMD)", _pkg_version("vmdpy")), ("xgboost", xgboost.__version__),
    ("torch (CNN-BiLSTM-Attn, LSTM-Attn)", torch.__version__),
    ("scikit-learn", sklearn.__version__), ("scipy", scipy.__version__),
], columns=["Package", "Version"])
print(REPORT["Software_versions"].to_string(index=False))

import pickle, shutil

def zip_and_download(label):
    """Zip the whole results folder and download it to your computer (Colab)."""
    base = os.path.join(UPLOAD_DIR, f"{POLL_TAG}_{label}")
    zp = shutil.make_archive(base, "zip", OUT_DIR)
    print(f"Zipped -> {zp}  ({os.path.getsize(zp) / 1e6:.1f} MB)")
    if AUTO_DOWNLOAD:
        try:
            from google.colab import files
            files.download(zp)
            print("Download started (allow multiple downloads in the browser if asked).")
        except Exception as e:
            print("Automatic download not available here:", e)
    return zp


Device: cuda
                           Package      Version
                            Python      3.13.16
                             numpy        2.4.6
                            pandas        2.2.3
   statsmodels (STL, ARIMA, tests)       0.15.0
                       vmdpy (VMD)          0.2
                           xgboost        3.4.1
torch (CNN-BiLSTM-Attn, LSTM-Attn) 2.14.1+cu130
                      scikit-learn        1.6.1
                             scipy       1.16.3


In [ ]:

raw = pd.read_csv(FILE_PATH)
n_rows_raw = len(raw)
raw["Date"] = pd.to_datetime(raw[["year", "month", "day", "hour"]], errors="coerce")
raw = raw.rename(columns={POLLUTANT: "Value"})[["Date", "Value"]]
DATE_FORMAT_USED = "separate year/month/day/hour columns (UCI Beijing Multi-Site)"
print("Station:", STATION, "| rows:", n_rows_raw)
n_bad_dates = int(raw["Date"].isna().sum())
raw["Value"] = pd.to_numeric(raw["Value"], errors="coerce")
raw = raw.dropna(subset=["Date"])
raw = raw[(raw["Date"] >= pd.Timestamp(DATE_START)) & (raw["Date"] <= pd.Timestamp(DATE_END))]
assert len(raw) > 1000, f"Too few rows in {DATE_START}..{DATE_END} - check DATE_COL / date format"
raw = raw.groupby("Date", as_index=False)["Value"].mean().sort_values("Date")

grid = pd.date_range(raw["Date"].min().floor("h"), raw["Date"].max().floor("h"), freq="h")
s = raw.set_index("Date")["Value"].reindex(grid)
observed = s.notna().values


isna = s.isna().astype(int).values
gap_lengths, run = [], 0
for v in isna:
    if v: run += 1
    elif run: gap_lengths.append(run); run = 0
if run: gap_lengths.append(run)
gap_lengths = np.array(gap_lengths) if gap_lengths else np.array([0])


s_filled = s.interpolate(method="linear", limit_area="inside").ffill().bfill()
y_all = s_filled.values.astype(float)
dates = grid
n = len(y_all)

per_year = pd.DataFrame({"year": grid.year, "obs": observed}).groupby("year")["obs"].agg(["size", "sum"])
per_year["capture_rate_%"] = 100 * per_year["sum"] / per_year["size"]
per_year = per_year.rename(columns={"size": "expected_hours", "sum": "observed_hours"}).reset_index()
REPORT["Data_capture_by_year"] = per_year

obs_vals = s.dropna()
REPORT["Data_summary"] = pd.DataFrame([
    ("Pollutant", POLLUTANT), ("Rows in file", n_rows_raw), ("Unparseable timestamps dropped", n_bad_dates),
    ("Date format detected", DATE_FORMAT_USED), ("Study period", f"{DATE_START} to {DATE_END}"),
    ("Start", str(grid[0])), ("End", str(grid[-1])),
    ("Expected hourly records", n), ("Observed records", int(observed.sum())),
    ("Missing records", int((~observed).sum())), ("Missing %", round(100 * (~observed).mean(), 3)),
    ("Overall data capture %", round(100 * observed.mean(), 3)),
    ("Number of gaps", int((gap_lengths > 0).sum())), ("Longest gap (h)", int(gap_lengths.max())),
    (f"Gaps <= {MAX_GAP_INTERP} h", int(((gap_lengths > 0) & (gap_lengths <= MAX_GAP_INTERP)).sum())),
    ("Negative values", int((obs_vals < 0).sum())),
    ("Mean", obs_vals.mean()), ("SD", obs_vals.std()), ("Min", obs_vals.min()),
    ("Median", obs_vals.median()), ("P95", obs_vals.quantile(0.95)), ("Max", obs_vals.max()),
    ("Missing-value handling", f"linear interpolation for continuity; imputed hours excluded as targets"),
], columns=["Item", "Value"])
print(REPORT["Data_summary"].to_string(index=False))


train_end = int(n * TRAIN_FRAC)
val_end   = int(n * (TRAIN_FRAC + VAL_FRAC))
first_origin = DECOMP_WINDOW - 1 + LOOKBACK - 1


fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(dates, s.values, lw=0.4, color="k")
ax.axvspan(dates[train_end], dates[val_end - 1], color="orange", alpha=0.2, label="Validation")
ax.axvspan(dates[val_end], dates[-1], color="red", alpha=0.15, label="Test")
ax.set_ylabel(f"{POLLUTANT} (µg/m³)"); ax.set_title(f"Hourly {POLLUTANT}, {STATION}, Beijing (raw observations)")
ax.legend(); fig.tight_layout(); fig.savefig(f"{OUT_DIR}/raw_series_{POLLUTANT}.png", dpi=300); plt.close(fig)


Station: Nongzhanguan | rows: 35064
                          Item                                                                  Value
                     Pollutant                                                                    NO2
                  Rows in file                                                                  35064
Unparseable timestamps dropped                                                                      0
          Date format detected          separate year/month/day/hour columns (UCI Beijing Multi-Site)
                  Study period                                   2013-03-01 00:00 to 2017-02-28 23:00
                         Start                                                    2013-03-01 00:00:00
                           End                                                    2017-02-28 23:00:00
       Expected hourly records                                                                  35064
              Observed records                

In [ ]:

def _decomp_chunk(y, origins, W, stl_params, stl_fit, vmd_params, K):
    rows = []
    for t in origins:
        w = y[t - W + 1:t + 1]
        r = STL(w, **stl_params).fit(**stl_fit)
        row = [r.trend[-1], r.seasonal[-1], r.resid[-1]]
        if K:
            u, _, _ = VMD(r.resid, vmd_params["alpha"], vmd_params["tau"], K,
                          vmd_params["DC"], vmd_params["init"], vmd_params["tol"])
            row += list(u[:, -1])
        rows.append(row)
    return np.asarray(rows)

def select_vmd_k():
    """K selection on TRAINING windows only: centre-frequency observation + reconstruction error."""
    rng = np.random.default_rng(SEED)
    cand = np.arange(DECOMP_WINDOW - 1, train_end)
    ts = rng.choice(cand, size=min(VMD_K_SAMPLE_WINDOWS, len(cand)), replace=False)
    resids = [STL(y_all[t - DECOMP_WINDOW + 1:t + 1], **STL_PARAMS).fit(**STL_FIT).resid for t in ts]
    rows = []
    for K in VMD_K_RANGE:
        gaps, errs, cfs = [], [], []
        for r in resids:
            u, _, om = VMD(r, VMD_PARAMS["alpha"], VMD_PARAMS["tau"], K,
                           VMD_PARAMS["DC"], VMD_PARAMS["init"], VMD_PARAMS["tol"])
            cf = np.sort(om[-1]); cfs.append(cf)
            gaps.append(np.min(np.diff(cf)))
            errs.append(np.linalg.norm(r[:u.shape[1]] - u.sum(0)) / np.linalg.norm(r))
        rows.append(dict(K=K, mean_min_cf_gap=np.mean(gaps), mean_rel_recon_error=np.mean(errs),
                         mean_centre_freqs=np.round(np.mean(cfs, 0), 4).tolist()))
    tab = pd.DataFrame(rows)
    ok = tab[tab["mean_min_cf_gap"] >= VMD_CF_GAP_MIN]
    K_sel = int(ok["K"].max()) if len(ok) else int(tab["K"].min())
    tab["criterion_choice"] = tab["K"] == K_sel
    return K_sel, tab

need_stl = any(tr != "RAW" for tr in TRACKS)
need_vmd = "STL-VMD" in TRACKS
TIMING = []

if need_vmd:
    K_auto, k_tab = select_vmd_k()
    if VMD_K is None:
        VMD_K = K_auto
    k_tab["used"] = k_tab["K"] == VMD_K
    REPORT["VMD_K_selection"] = k_tab
    print(k_tab.to_string(index=False))
    print("VMD K =", VMD_K)

DECOMP = {}
if need_stl:
    K_use = VMD_K if need_vmd else 0
    poll_tag = POLLUTANT.replace(".", "")
    cache_name = f"causal_decomp_{poll_tag}_W{DECOMP_WINDOW}_K{K_use}.npy"
    old_name = f"causal_decomp_W{DECOMP_WINDOW}_K{K_use}.npy"
    cache = f"{OUT_DIR}/{cache_name}"
    origins = np.arange(DECOMP_WINDOW - 1, n)

    def cache_ok(path):
        """A cache is used only if it matches THIS series: same length and identical values at test origins."""
        try:
            c = np.load(path)
        except Exception:
            return None
        if c.shape != (len(origins), 3 + K_use):
            return None
        idx = np.linspace(0, len(origins) - 1, 4).astype(int)
        chk = _decomp_chunk(y_all, origins[idx], DECOMP_WINDOW, STL_PARAMS, STL_FIT, VMD_PARAMS, K_use)
        return c if np.allclose(chk, c[idx], rtol=1e-5, atol=1e-6) else None

    comp = None
    t0 = time.time()
    for cand in [cache, f"/content/{cache_name}", f"/content/{old_name}", f"{OUT_DIR}/{old_name}",
                 f"/content/Beijing_Nongzhanguan_NO2_{POLLUTANT}/{old_name}"]:
        if os.path.exists(cand):
            comp = cache_ok(cand)
            if comp is not None:
                print("Using verified cached decomposition:", cand)
                if cand != cache: np.save(cache, comp)
                break
            print("Cache does not match this series (ignored):", cand)
    if comp is None:
        chunks = np.array_split(origins, max(1, (os.cpu_count() or 1) * 8))
        parts = Parallel(n_jobs=N_JOBS, verbose=5)(
            delayed(_decomp_chunk)(y_all[:c[-1] + 1], c, DECOMP_WINDOW, STL_PARAMS, STL_FIT, VMD_PARAMS, K_use)
            for c in chunks if len(c))
        comp = np.vstack(parts); np.save(cache, comp)
    t_dec = time.time() - t0
    full = np.full((n, comp.shape[1]), np.nan); full[origins] = comp
    DECOMP["trend"], DECOMP["seasonal"], DECOMP["resid"] = full[:, 0], full[:, 1], full[:, 2]
    for k in range(K_use): DECOMP[f"VMD_{k+1}"] = full[:, 3 + k]
    TIMING.append(dict(Track="STL(+VMD) causal decomposition", Model="-", Stage="decomposition",
                       Seconds=t_dec, ms_per_origin=1000 * t_dec / len(origins)))

REPORT["STL_parameters"] = pd.DataFrame(
    [(k, v) for k, v in STL_PARAMS.items()] + [(k, v) for k, v in STL_FIT.items()] +
    [("causal window (h)", DECOMP_WINDOW), ("basis", "period = diurnal cycle; seasonal=7 (Cleveland default); "
      "trend = next odd int of 1.5*period/(1-1.5/seasonal); low_pass = next odd > period; "
      "robust iterations = statsmodels robust defaults; identical for all pollutants")],
    columns=["Parameter", "Value"])
REPORT["VMD_parameters"] = pd.DataFrame(
    [("K", VMD_K)] + [(k, v) for k, v in VMD_PARAMS.items()] + [("input", "STL remainder of each causal window")],
    columns=["Parameter", "Value"])

zip_and_download("checkpoint")


 K  mean_min_cf_gap  mean_rel_recon_error                                                                             mean_centre_freqs  criterion_choice  used
 2         0.111433              0.519096                                                                               [0.0256, 0.137]             False False
 3         0.063003              0.438789                                                                      [0.0228, 0.0872, 0.2624]             False False
 4         0.046990              0.372879                                                               [0.0212, 0.069, 0.1734, 0.3379]             False False
 5         0.040712              0.320069                                                       [0.0205, 0.0626, 0.1354, 0.2518, 0.386]             False False
 6         0.036573              0.272286                                              [0.0198, 0.0577, 0.1159, 0.1966, 0.2976, 0.4059]             False False
 7         0.034504              0.23970

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 2 concurrent workers.


Zipped -> /content/Nongzhanguan_NO2_checkpoint.zip  (3.4 MB)


[Parallel(n_jobs=-1)]: Done  16 out of  16 | elapsed: 94.2min finished


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started (allow multiple downloads in the browser if asked).


'/content/Nongzhanguan_NO2_checkpoint.zip'

In [ ]:

def stationarity(x):
    adf_stat, adf_p = adfuller(x, autolag="AIC")[:2]
    kpss_stat, kpss_p = kpss(x, regression="c", nlags="auto")[:2]
    return adf_stat, adf_p, kpss_stat, kpss_p

def arima_grid(x_train, pq_max, label):
    adf_s, adf_p, kp_s, kp_p = stationarity(x_train)
    d = 0 if (adf_p < 0.05 and kp_p > 0.05) else 1
    rows = []
    for p in range(pq_max + 1):
        for q in range(pq_max + 1):
            try:
                f = ARIMA(x_train, order=(p, d, q)).fit()
                rows.append(dict(series=label, p=p, d=d, q=q, AIC=f.aic, BIC=f.bic))
            except Exception as e:
                rows.append(dict(series=label, p=p, d=d, q=q, AIC=np.nan, BIC=np.nan))
    g = pd.DataFrame(rows)
    best = g.loc[g[ARIMA_CRITERION.upper()].idxmin()]
    order = (int(best.p), int(best.d), int(best.q))
    st = pd.DataFrame([dict(series=label, ADF_stat=adf_s, ADF_p=adf_p, KPSS_stat=kp_s, KPSS_p=kp_p,
                            chosen_d=d, rule="d=0 if ADF rejects unit root (p<0.05) and KPSS does not reject stationarity (p>0.05)")])
    return order, g, st

def arima_diagnostics(fit, label):
    r = pd.Series(fit.resid).iloc[max(fit.model.k_ar, 1):]
    lb = acorr_ljungbox(r, lags=[24, 48, 168], return_df=True).reset_index().rename(columns={"index": "lag"})
    lb.insert(0, "series", label)
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
    plot_acf(r, lags=72, ax=ax[0]); plot_pacf(r, lags=72, ax=ax[1], method="ywm")
    ax[0].set_title(f"ACF of ARIMA residuals ({label})"); ax[1].set_title(f"PACF of ARIMA residuals ({label})")
    fig.tight_layout(); fig.savefig(f"{OUT_DIR}/arima_resid_acf_pacf_{label}_{POLLUTANT}.png", dpi=300); plt.close(fig)
    return lb

ARIMA_TABLES, ARIMA_LB, ARIMA_ST = [], [], []
if "STL-ARIMA" in TRACKS:
    rr = DECOMP["resid"]
    i0 = DECOMP_WINDOW - 1
    r_train = rr[i0:train_end]
    t0 = time.time()
    order, g, st = arima_grid(r_train, ARIMA_PQ_MAX, "STL remainder")
    fit = ARIMA(r_train, order=order).fit()
    t_fit = time.time() - t0
    ARIMA_TABLES.append(g); ARIMA_ST.append(st); ARIMA_LB.append(arima_diagnostics(fit, "STL_remainder"))

    full_res = fit.apply(rr[i0:])
    fitted = np.asarray(full_res.fittedvalues)
    nxt = np.append(fitted[1:], np.asarray(full_res.forecast(1)))
    DECOMP["ARIMA_fit"] = np.r_[np.full(i0, np.nan), fitted]
    DECOMP["ARIMA_innov"] = np.r_[np.full(i0, np.nan), rr[i0:] - fitted]
    DECOMP["ARIMA_next"] = np.r_[np.full(i0, np.nan), nxt]
    TIMING.append(dict(Track="STL-ARIMA", Model=f"ARIMA{order}", Stage="order search + fit", Seconds=t_fit))
    print("STL-ARIMA order:", order)
    STL_ARIMA_ORDER = order


STL-ARIMA order: (2, 0, 3)


In [ ]:

def metrics(y, p):
    e = y - p
    return dict(N=len(y), MAE=mean_absolute_error(y, p), RMSE=np.sqrt(mean_squared_error(y, p)),
                MASE=np.mean(np.abs(e)) / MASE_SCALE, R2=r2_score(y, p), PCC=np.corrcoef(y, p)[0, 1],
                Bias=np.mean(p - y), SD_ratio=np.std(p) / np.std(y),
                CRMSE=np.sqrt(np.mean(((p - p.mean()) - (y - y.mean())) ** 2)))

def dm_test(y, p1, p2, h=None, loss="squared"):
    """Diebold-Mariano with Harvey-Leybourne-Newbold correction. negative stat -> p1 better."""
    h = HORIZON if h is None else h
    if loss == "squared": d = (y - p1) ** 2 - (y - p2) ** 2
    else:                 d = np.abs(y - p1) - np.abs(y - p2)
    T = len(d); dbar = d.mean()
    gam = [np.mean((d[k:] - dbar) * (d[:T - k] - dbar)) for k in range(h)]
    var = (gam[0] + 2 * sum(gam[1:])) / T
    dm = dbar / np.sqrt(var)
    dm *= np.sqrt((T + 1 - 2 * h + h * (h - 1) / T) / T)
    p = 2 * stats.t.sf(np.abs(dm), df=T - 1)
    return dm, p

def residual_diag(y, p, y_peak_thr):
    e = y - p
    lb = acorr_ljungbox(e, lags=[24], return_df=True)
    X_bp = np.column_stack([np.ones_like(p), p])
    bp_p = het_breuschpagan(e, X_bp)[1]
    arch_p = het_arch(e, nlags=24)[1]
    pk = y >= y_peak_thr
    m_rel = np.abs(y) >= RELRES_MIN
    rel = e[m_rel] / y[m_rel]
    return dict(mean_error=e.mean(), SD_error=e.std(ddof=1), skewness=stats.skew(e),
                excess_kurtosis=stats.kurtosis(e), JarqueBera_p=stats.jarque_bera(e).pvalue,
                LjungBox_Q24=lb["lb_stat"].iloc[0], LjungBox_p24=lb["lb_pvalue"].iloc[0],
                BreuschPagan_p=bp_p, ARCH_LM24_p=arch_p,
                peak_threshold=y_peak_thr, n_peak=int(pk.sum()),
                peak_mean_error=e[pk].mean(), peak_MAE=np.abs(e[pk]).mean(),
                peak_underpred_share=(e[pk] > 0).mean(), nonpeak_MAE=np.abs(e[~pk]).mean(),
                rel_resid_median=np.median(rel), rel_resid_IQR=np.subtract(*np.percentile(rel, [75, 25])),
                rel_resid_excluded=int((~m_rel).sum()))

def uncertainty(e_val, e_test, rmse_test):
    """R2-13: explicit definitions.
    sigma_e = SD of test errors; U95 = 1.96*sqrt(sigma_e^2 + RMSE^2) (expanded uncertainty);
    PI95 = split-conformal interval: half-width q = 95% quantile of |validation errors|;
    PICP = share of test observations inside y_hat +/- q."""
    q = np.quantile(np.abs(e_val), 0.95)
    sd = e_test.std(ddof=1)
    return dict(sigma_e=sd, U95=1.96 * np.sqrt(sd ** 2 + rmse_test ** 2),
                PI95_halfwidth=q, PI95_width=2 * q, PICP_test=np.mean(np.abs(e_test) <= q))


class Attention(nn.Module):
    def __init__(self, dim):
        super().__init__(); self.fc = nn.Linear(dim, 1); self.last_w = None
    def forward(self, x):
        w = torch.softmax(self.fc(x), dim=1); self.last_w = w.detach()
        return torch.sum(w * x, dim=1)

class CNN_BiLSTM_Attn(nn.Module):
    def __init__(self, input_dim, filters=32, kernel=3, lstm_units=32, dropout=0.2, **_):
        super().__init__()
        self.conv = nn.Conv1d(input_dim, filters, kernel, padding=kernel // 2)
        self.act = nn.ReLU()
        self.lstm = nn.LSTM(filters, lstm_units, batch_first=True, bidirectional=True)
        self.attn = Attention(2 * lstm_units); self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(2 * lstm_units, 1)
    def forward(self, x):
        x = self.act(self.conv(x.permute(0, 2, 1))).permute(0, 2, 1)
        x, _ = self.lstm(x)
        return self.fc(self.drop(self.attn(x)))

class LSTMAttention(nn.Module):
    def __init__(self, input_dim, hidden=128, layers=2, dropout=0.3, **_):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden, num_layers=layers, batch_first=True,
                            dropout=dropout if layers > 1 else 0.0)
        self.norm = nn.LayerNorm(hidden); self.attn = Attention(hidden)
        self.drop = nn.Dropout(dropout); self.fc = nn.Linear(hidden, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(self.drop(self.attn(self.norm(out))))

DL_GRID = {
    "CNN-BiLSTM-Attn": [dict(filters=32, kernel=3, lstm_units=32, dropout=0.2, lr=1e-3),
                        dict(filters=64, kernel=3, lstm_units=64, dropout=0.2, lr=1e-3),
                        dict(filters=64, kernel=3, lstm_units=64, dropout=0.2, lr=5e-4)],
    "LSTM-Attn":       [dict(hidden=64, layers=2, dropout=0.3, lr=1e-3),
                        dict(hidden=128, layers=2, dropout=0.3, lr=1e-3),
                        dict(hidden=128, layers=2, dropout=0.3, lr=5e-4)],
}
DL_CLASS = {"CNN-BiLSTM-Attn": CNN_BiLSTM_Attn, "LSTM-Attn": LSTMAttention}
ADAM = dict(betas=(0.9, 0.999), eps=1e-8, weight_decay=1e-5)
HUBER_DELTA = 1.0

def predict_dl(model, X, bs=4096):
    model.eval(); preds, attns = [], []
    with torch.no_grad():
        for i in range(0, len(X), bs):
            xb = torch.tensor(X[i:i + bs], device=DEVICE)
            preds.append(model(xb).cpu().numpy().ravel())
            attns.append(model.attn.last_w.squeeze(-1).cpu().numpy())
    return np.concatenate(preds), np.concatenate(attns)

def train_dl(name, cfg, Xtr, ytr, Xva, yva):
    torch.manual_seed(SEED)
    model = DL_CLASS[name](Xtr.shape[2], **cfg).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=cfg["lr"], **ADAM)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=5, factor=0.5)
    loss_fn = nn.HuberLoss(delta=HUBER_DELTA)
    g = torch.Generator().manual_seed(SEED)
    loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(torch.tensor(Xtr), torch.tensor(ytr)),
                                         batch_size=BATCH_SIZE, shuffle=True, generator=g)

    Xva_t, yva_t = torch.tensor(Xva, device=DEVICE), torch.tensor(yva, device=DEVICE)
    best, best_state, best_ep, wait = np.inf, None, 0, 0
    for ep in range(DL_MAX_EPOCHS):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(); loss = loss_fn(model(xb), yb); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        model.eval()
        with torch.no_grad():
            vl = float(sum(loss_fn(model(Xva_t[i:i+4096]), yva_t[i:i+4096]).item() * len(yva_t[i:i+4096])
                           for i in range(0, len(yva_t), 4096)) / len(yva_t))
        sched.step(vl)
        if vl < best - 1e-7:
            best, best_ep, wait = vl, ep + 1, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= DL_PATIENCE: break
    model.load_state_dict(best_state)
    return model, best_ep, ep + 1

def sample_xgb(rng):
    return dict(max_depth=int(rng.integers(3, 11)), learning_rate=float(10 ** rng.uniform(-1.7, -0.7)),
                subsample=float(rng.uniform(0.6, 1.0)), colsample_bytree=float(rng.uniform(0.5, 1.0)),
                min_child_weight=float(rng.integers(1, 11)), gamma=float(rng.uniform(0, 2)),
                reg_alpha=float(10 ** rng.uniform(-3, 1)), reg_lambda=float(10 ** rng.uniform(-2, 1)))

XGB_SPACE = ("max_depth U{3..10}; learning_rate log-U[0.02,0.2]; subsample U[0.6,1]; colsample_bytree U[0.5,1]; "
             "min_child_weight U{1..10}; gamma U[0,2]; reg_alpha log-U[1e-3,10]; reg_lambda log-U[1e-2,10]; "
             f"n_estimators <= {XGB_MAX_TREES} with early stopping (50 rounds) on the validation block; "
             f"random search, {XGB_TRIALS} trials, selection by validation RMSE")

def tune_xgb(Xtr, ytr, Xva, yva):
    rng = np.random.default_rng(SEED); best = (np.inf, None, None); rows = []
    for i in range(XGB_TRIALS):
        p = sample_xgb(rng)
        m = XGBRegressor(n_estimators=XGB_MAX_TREES, early_stopping_rounds=50, objective="reg:squarederror",
                         eval_metric="rmse", tree_method="hist", device=XGB_DEVICE, random_state=SEED,
                         n_jobs=-1, **p)
        m.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
        rm = np.sqrt(mean_squared_error(yva, m.predict(Xva)))
        rows.append({**p, "best_iteration": m.best_iteration, "val_RMSE": rm})
        if rm < best[0]: best = (rm, m, {**p, "n_estimators(best_iteration+1)": m.best_iteration + 1})
    return best[1], best[2], pd.DataFrame(rows)


In [ ]:

BASE_REPORT, BASE_TIMING = dict(REPORT), list(TIMING)
BASE_ARIMA = (list(ARIMA_TABLES), list(ARIMA_ST), list(ARIMA_LB))

def run_horizon(h):
    global A, ARIMA_LB, ARIMA_ST, ARIMA_TABLES, ATTN, BASE, DL_HP, FIG_DIR, HORIZON, MASE_SCALE, MET, PRED, REPORT, SEASON, SET, TARGET_DATES, TIMING, TRACK_FEATS, WEIGHTS, X3, XGB_HP, XGB_SEARCH, Xf, Y, Ys, _, a_all, a_te, ax, axes, base, best, best_ep, build_track, c, cfg, cfg_grid, ck, cols, diag, dm, dow, e, ent, fh, fig, fit_raw, fv, g_raw, hour, hp, inv, k, keep, lag_cols, loss, m, m_te, m_tr, m_va, mean_w, model, month, mse_v, n0, name, nm, order_raw, origins_all, p, p_all, pred_df, pv, r, ran, ref_key, ref_name, rm, rows, search, seas, sn, split_info, st, st_raw, t0, test_months, tgt, tr_idx, track, unc, v, w, xgb, xlsx, y_peak_thr, y_scaler, y_tr_raw, ys
    HORIZON = h
    xlsx = f"{OUT_DIR}/{POLLUTANT}_h{HORIZON}_Beijing_Nongzhanguan_NO2_results.xlsx"
    if SKIP_DONE and os.path.exists(xlsx):
        print(f"#### h = {HORIZON}: already finished ({xlsx}) - skipped ####")
        return
    print(f"\n################ HORIZON h = {HORIZON} ################")
    REPORT, TIMING = dict(BASE_REPORT), list(BASE_TIMING)
    ARIMA_TABLES, ARIMA_ST, ARIMA_LB = [list(x) for x in BASE_ARIMA]
    FIG_DIR = os.path.join(OUT_DIR, f"figures_h{HORIZON}"); os.makedirs(FIG_DIR, exist_ok=True)
    ys = pd.Series(y_all)
    hour, dow, month = dates.hour.values, dates.dayofweek.values, dates.month.values
    BASE = {
        "y": y_all,
        "roll7": ys.rolling(7).mean().values,
        "roll14": ys.rolling(14).mean().values,
        "diff1": ys.diff().values,
        "hour_sin": np.sin(2*np.pi*hour/24), "hour_cos": np.cos(2*np.pi*hour/24),
        "dow_sin": np.sin(2*np.pi*dow/7), "dow_cos": np.cos(2*np.pi*dow/7),
        "month_sin": np.sin(2*np.pi*(month-1)/12), "month_cos": np.cos(2*np.pi*(month-1)/12),
    }
    TRACK_FEATS = {
        "RAW": [],
        "STL": ["trend", "seasonal", "resid"],
        "STL-VMD": ["trend", "seasonal"] + [f"VMD_{k+1}" for k in range(VMD_K or 0)],
        "STL-ARIMA": ["trend", "seasonal", "ARIMA_fit", "ARIMA_innov", "ARIMA_next"],
    }


    origins_all = np.arange(first_origin, n - HORIZON)
    tgt = origins_all + HORIZON
    keep = observed[tgt]
    origins_all, tgt = origins_all[keep], tgt[keep]
    SET = np.where(tgt < train_end, "train", np.where(tgt < val_end, "val", "test"))

    Y = y_all[tgt]
    TARGET_DATES = dates[tgt]

    def split_info():
        rows = []
        for nm in ["train", "val", "test"]:
            m = SET == nm
            rows.append(dict(set=nm, n_samples=int(m.sum()), first_target=str(TARGET_DATES[m][0]),
                             last_target=str(TARGET_DATES[m][-1]),
                             months_covered=int(pd.Series(TARGET_DATES[m].month).nunique())))
        df = pd.DataFrame(rows)
        df["lookback_h"], df["horizon_h"], df["decomp_window_h"] = LOOKBACK, HORIZON, DECOMP_WINDOW
        return df
    REPORT["Split_info"] = split_info()
    print(REPORT["Split_info"].to_string(index=False))
    test_months = pd.Series(TARGET_DATES[SET == "test"].month).value_counts().sort_index()
    REPORT["Test_month_coverage"] = test_months.rename_axis("month").reset_index(name="n_test_samples")

    def build_track(track):
        cols = list(BASE) + TRACK_FEATS[track]
        F = np.column_stack([BASE[c] if c in BASE else DECOMP[c] for c in cols]).astype(np.float32)
        assert not np.isnan(F[first_origin - LOOKBACK + 1:]).any(), f"NaN in features for {track}"
        sc = MinMaxScaler().fit(F[first_origin - LOOKBACK + 1:train_end])
        Fs = np.nan_to_num(sc.transform(np.nan_to_num(F)), nan=0.0).astype(np.float32)
        win = np.lib.stride_tricks.sliding_window_view(Fs, LOOKBACK, axis=0)
        X3 = np.transpose(win[origins_all - LOOKBACK + 1], (0, 2, 1)).copy()
        return cols, X3

    y_scaler = MinMaxScaler().fit(Y[SET == "train"].reshape(-1, 1))
    Ys = y_scaler.transform(Y.reshape(-1, 1)).astype(np.float32)
    inv = lambda a: y_scaler.inverse_transform(np.asarray(a).reshape(-1, 1)).ravel()


    tr_idx = tgt[SET == "train"]
    MASE_SCALE = np.mean(np.abs(y_all[tr_idx] - y_all[tr_idx - HORIZON]))



    PRED = {}
    PRED[("BASELINE", "Naive persistence")] = y_all[origins_all]
    PRED[("BASELINE", "Seasonal naive (24 h)")] = y_all[origins_all + HORIZON - 24]
    if HORIZON == 1:
        t0 = time.time()
        y_tr_raw = y_all[:train_end]
        order_raw, g_raw, st_raw = arima_grid(y_tr_raw, RAW_ARIMA_PQ_MAX, "raw series")
        fit_raw = ARIMA(y_tr_raw, order=order_raw).fit()
        ARIMA_TABLES.append(g_raw); ARIMA_ST.append(st_raw); ARIMA_LB.append(arima_diagnostics(fit_raw, "raw_series"))
        fv = np.asarray(fit_raw.apply(y_all).fittedvalues)
        PRED[("BASELINE", f"Raw ARIMA{order_raw}")] = fv[origins_all + 1]
        TIMING.append(dict(Track="BASELINE", Model=f"Raw ARIMA{order_raw}", Stage="order search + fit", Seconds=time.time() - t0))
    if ARIMA_TABLES:
        REPORT["ARIMA_order_grid"] = pd.concat(ARIMA_TABLES, ignore_index=True)
        REPORT["ARIMA_stationarity"] = pd.concat(ARIMA_ST, ignore_index=True)
        REPORT["ARIMA_LjungBox"] = pd.concat(ARIMA_LB, ignore_index=True)



    m_tr, m_va, m_te = SET == "train", SET == "val", SET == "test"
    XGB_HP, DL_HP, XGB_SEARCH, WEIGHTS, ATTN = [], [], [], [], []

    for track in TRACKS:
        print(f"\n================ {track} ================")
        ck = f"{OUT_DIR}/ckpt_h{HORIZON}_{track}.pkl"
        if SKIP_DONE and os.path.exists(ck):
            with open(ck, "rb") as fh:
                c = pickle.load(fh)
            PRED.update(c["PRED"]); XGB_HP += c["XGB_HP"]; DL_HP += c["DL_HP"]; XGB_SEARCH += c["XGB_SEARCH"]
            WEIGHTS += c["WEIGHTS"]; ATTN += c["ATTN"]; TIMING += c["TIMING"]
            print(f"{track}: loaded from checkpoint (already trained)")
            continue
        n0 = dict(XGB_HP=len(XGB_HP), DL_HP=len(DL_HP), XGB_SEARCH=len(XGB_SEARCH),
                  WEIGHTS=len(WEIGHTS), ATTN=len(ATTN), TIMING=len(TIMING))
        cols, X3 = build_track(track)
        Xf = X3.reshape(len(X3), -1)

        t0 = time.time()
        xgb, hp, search = tune_xgb(Xf[m_tr], Y[m_tr], Xf[m_va], Y[m_va])
        TIMING.append(dict(Track=track, Model="XGBoost", Stage="tuning + training", Seconds=time.time() - t0))
        t0 = time.time(); PRED[(track, "XGBoost")] = xgb.predict(Xf)
        TIMING.append(dict(Track=track, Model="XGBoost", Stage="inference (all samples)", Seconds=time.time() - t0,
                           ms_per_1000=1000 * (time.time() - t0) / len(Xf) * 1000))
        XGB_HP.append({"Track": track, **hp, "objective": "reg:squarederror", "tree_method": "hist"})
        search.insert(0, "Track", track); XGB_SEARCH.append(search)
        print("XGB best:", hp)

        for name in ["CNN-BiLSTM-Attn", "LSTM-Attn"]:
            cfg_grid = DL_GRID[name] if TUNE_DL else DL_GRID[name][:1]
            best = (np.inf, None, None)
            t0 = time.time()
            for cfg in cfg_grid:
                model, best_ep, ran = train_dl(name, cfg, X3[m_tr], Ys[m_tr], X3[m_va], Ys[m_va])
                pv, _ = predict_dl(model, X3[m_va])
                rm = np.sqrt(mean_squared_error(Y[m_va], inv(pv)))
                DL_HP.append({"Track": track, "Model": name, **cfg, "best_epoch": best_ep, "epochs_run": ran,
                              "val_RMSE": rm, "selected": False})
                if rm < best[0]: best = (rm, model, len(DL_HP) - 1)
            DL_HP[best[2]]["selected"] = True
            TIMING.append(dict(Track=track, Model=name, Stage=f"tuning ({len(cfg_grid)} configs) + training", Seconds=time.time() - t0))
            t0 = time.time(); p_all, a_all = predict_dl(best[1], X3)
            TIMING.append(dict(Track=track, Model=name, Stage="inference (all samples)", Seconds=time.time() - t0,
                               ms_per_1000=1000 * (time.time() - t0) / len(X3) * 1000))
            PRED[(track, name)] = inv(p_all)
            a_te = a_all[m_te]
            mean_w = a_te.mean(0)
            ent = -(a_te * np.log(a_te + 1e-12)).sum(1).mean() / np.log(LOOKBACK)
            ATTN.append({"Track": track, "Model": name, "normalised_entropy": ent,
                         **{f"lag_{LOOKBACK-1-i}h": mean_w[i] for i in range(LOOKBACK)}})
            print(f"{name}: val RMSE {best[0]:.3f}")

        base = ["CNN-BiLSTM-Attn", "LSTM-Attn", "XGBoost"]
        mse_v = np.array([mean_squared_error(Y[m_va], PRED[(track, b)][m_va]) for b in base])
        w = (1 / mse_v) / (1 / mse_v).sum()
        WEIGHTS.append({"Track": track, **{f"w_{b}": wi for b, wi in zip(base, w)},
                        **{f"val_MSE_{b}": v for b, v in zip(base, mse_v)}, "rule": "w_i = (1/MSE_i,val) / sum_j (1/MSE_j,val)"})
        PRED[(track, "Ensemble_Weighted")] = sum(wi * PRED[(track, b)] for wi, b in zip(w, base))
        PRED[(track, "Ensemble_Equal")] = sum(PRED[(track, b)] for b in base) / 3
        with open(ck, "wb") as fh:
            pickle.dump(dict(PRED={k: v for k, v in PRED.items() if k[0] == track},
                             XGB_HP=XGB_HP[n0["XGB_HP"]:], DL_HP=DL_HP[n0["DL_HP"]:], XGB_SEARCH=XGB_SEARCH[n0["XGB_SEARCH"]:],
                             WEIGHTS=WEIGHTS[n0["WEIGHTS"]:], ATTN=ATTN[n0["ATTN"]:], TIMING=TIMING[n0["TIMING"]:]), fh)
        print(f"{track}: checkpoint saved -> {ck}")

    REPORT["XGB_hyperparameters"] = pd.DataFrame(XGB_HP).assign(search_space=XGB_SPACE)
    REPORT["XGB_search_trials"] = pd.concat(XGB_SEARCH, ignore_index=True)
    REPORT["DL_hyperparameters"] = pd.DataFrame(DL_HP).assign(
        optimizer="Adam(beta1=0.9, beta2=0.999, eps=1e-8, weight_decay=1e-5)", loss=f"Huber(delta={HUBER_DELTA})",
        batch_size=BATCH_SIZE, max_epochs=DL_MAX_EPOCHS, early_stopping_patience=DL_PATIENCE,
        lr_schedule="ReduceLROnPlateau(factor=0.5, patience=5) on validation loss", grad_clip=1.0,
        lookback=LOOKBACK, selection="validation RMSE")
    REPORT["DL_architecture"] = pd.DataFrame([
        ("CNN-BiLSTM-Attn", "Conv1D x1 (kernel 3, 'same' padding, ReLU, no pooling) -> BiLSTM x1 -> additive attention "
         "(Linear(2u,1)+softmax over time) -> Dropout -> Linear(2u,1)"),
        ("LSTM-Attn", "LSTM x layers (inter-layer dropout) -> LayerNorm -> additive attention (Linear(h,1)+softmax) "
         "-> Dropout -> Linear(h,1)"),
        ("XGBoost", "gradient-boosted trees on the flattened LOOKBACK x features window; NOT trained with Adam/Huber"),
    ], columns=["Model", "Architecture"])
    REPORT["Ensemble_weights"] = pd.DataFrame(WEIGHTS)
    REPORT["Attention_weights_test"] = pd.DataFrame(ATTN)



    y_peak_thr = np.percentile(Y[m_tr], PEAK_PCT)
    rows, diag, unc, dm = [], [], [], []
    for (track, model), p in PRED.items():
        for nm, m in [("train", m_tr), ("val", m_va), ("test", m_te)]:
            rows.append({"Track": track, "Model": model, "Set": nm, **metrics(Y[m], p[m])})
        diag.append({"Track": track, "Model": model, **residual_diag(Y[m_te], p[m_te], y_peak_thr)})
        rm = np.sqrt(mean_squared_error(Y[m_te], p[m_te]))
        unc.append({"Track": track, "Model": model, **uncertainty(Y[m_va] - p[m_va], Y[m_te] - p[m_te], rm)})
        if track != "BASELINE":
            for ref_key, ref_name in [(("BASELINE", "Naive persistence"), "Naive persistence"),
                                      (("RAW", model), f"RAW {model}")]:
                if ref_key in PRED and ref_key != (track, model):
                    for loss in ["squared", "absolute"]:
                        st, pv = dm_test(Y[m_te], p[m_te], PRED[ref_key][m_te], loss=loss)
                        dm.append(dict(Track=track, Model=model, Reference=ref_name, Loss=loss, DM_stat=st, p_value=pv,
                                       better=("model" if st < 0 else "reference") if pv < 0.05 else "no significant difference"))

    MET = pd.DataFrame(rows)
    SEASON = pd.Series(TARGET_DATES.month).map({12: "DJF", 1: "DJF", 2: "DJF", 3: "MAM", 4: "MAM", 5: "MAM",
                                                6: "JJA", 7: "JJA", 8: "JJA", 9: "SON", 10: "SON", 11: "SON"}).values
    seas = []
    for (track, model), p in PRED.items():
        for sn in ["DJF", "MAM", "JJA", "SON"]:
            m = m_te & (SEASON == sn)
            if m.sum() > 10:
                seas.append({"Track": track, "Model": model, "Season": sn, "N": int(m.sum()),
                             "MAE": mean_absolute_error(Y[m], p[m]), "RMSE": np.sqrt(mean_squared_error(Y[m], p[m]))})
    REPORT["Metrics_test_by_season"] = pd.DataFrame(seas)
    REPORT["Metrics_all_sets"] = MET
    for nm in ["train", "test"]:
        REPORT[f"Table_{nm}"] = MET[MET.Set == nm][["Track", "Model", "MAE", "RMSE", "MASE", "R2"]].reset_index(drop=True)
    REPORT["Residual_diagnostics_test"] = pd.DataFrame(diag)
    REPORT["Uncertainty_test"] = pd.DataFrame(unc)
    REPORT["DM_tests_test"] = pd.DataFrame(dm)
    REPORT["Timing"] = pd.DataFrame(TIMING)

    print("\nTEST SET (primary metrics: MAE, RMSE, MASE; R2 secondary)")
    print(REPORT["Table_test"].sort_values("MASE").to_string(index=False, float_format="%.4f"))



    for track in [t for t in TRACKS]:
        fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
        for ax, model in zip(axes, ["XGBoost", "Ensemble_Weighted", "CNN-BiLSTM-Attn"]):
            plot_acf(Y[m_te] - PRED[(track, model)][m_te], lags=72, ax=ax); ax.set_title(f"{track} {model}: test-error ACF")
        fig.tight_layout(); fig.savefig(f"{FIG_DIR}/test_error_acf_{track}_{POLLUTANT}.png", dpi=300); plt.close(fig)

    A = REPORT["Attention_weights_test"]
    lag_cols = [c for c in A.columns if c.startswith("lag_")]
    fig, ax = plt.subplots(figsize=(9, 4))
    for _, r in A.iterrows():
        ax.plot([int(c[4:-1]) for c in lag_cols], r[lag_cols].values, marker="o", ms=3, label=f"{r.Track} {r.Model}")
    ax.set_xlabel("Lag (hours before forecast origin)"); ax.set_ylabel("Mean attention weight (test)")
    ax.invert_xaxis(); ax.legend(fontsize=7, ncol=2); fig.tight_layout()
    fig.savefig(f"{FIG_DIR}/attention_by_lag_{POLLUTANT}.png", dpi=300); plt.close(fig)

    fig, axes = plt.subplots(1, len(TRACKS), figsize=(4 * len(TRACKS), 3.8), sharey=True)
    for ax, track in zip(np.atleast_1d(axes), TRACKS):
        e = Y[m_te] - PRED[(track, "Ensemble_Weighted")][m_te]
        ax.scatter(Y[m_te], e, s=2, alpha=0.3); ax.axhline(0, color="k", lw=0.8)
        ax.axvline(y_peak_thr, color="r", ls="--", lw=0.8, label=f"P{PEAK_PCT} train")
        ax.set_title(f"{track} weighted ensemble (test)"); ax.set_xlabel("Observed")
    np.atleast_1d(axes)[0].set_ylabel("Error (obs - pred)"); np.atleast_1d(axes)[0].legend()
    fig.tight_layout(); fig.savefig(f"{FIG_DIR}/errors_vs_observed_peaks_{POLLUTANT}.png", dpi=300); plt.close(fig)



    with pd.ExcelWriter(xlsx) as w:
        for k, v in REPORT.items():
            v.to_excel(w, sheet_name=k[:31], index=False)
    pred_df = pd.DataFrame({"Target_time": TARGET_DATES, "Origin_time": dates[origins_all], "Set": SET, "Actual": Y})
    for (track, model), p in PRED.items():
        pred_df[f"{track} | {model}"] = p
    pred_df.to_csv(f"{OUT_DIR}/{POLLUTANT}_h{HORIZON}_predictions_all_sets.csv", index=False)
    print("\nSaved:", xlsx)
    print("Saved:", f"{OUT_DIR}/{POLLUTANT}_h{HORIZON}_predictions_all_sets.csv")
    print("Figures in:", FIG_DIR)

print("run_horizon() is ready.")


run_horizon() is ready.


In [2]:

if 1 in HORIZONS:
    run_horizon(1)
    zip_and_download("checkpoint")
else:
    print("h = 1 skipped (not in HORIZONS)")


 
################ HORIZON h = 1 ################
  set  n_samples        first_target         last_target  months_covered  lookback_h  horizon_h  decomp_window_h
train      23676 2013-03-15 23:00:00 2015-12-18 15:00:00              12          24          1              336
  val       3472 2015-12-18 18:00:00 2016-05-12 18:00:00               6          24          1              336
 test       6867 2016-05-12 19:00:00 2017-02-28 23:00:00              10          24          1              336

================ RAW ================
XGB best: {'max_depth': 4, 'learning_rate': 0.02783070377990749, 'subsample': 0.6413611870890207, 'colsample_bytree': 0.793822286088856, 'min_child_weight': 4.0, 'gamma': 0.3411859370737722, 'reg_alpha': 5.017420195002703, 'reg_lambda': 0.553583859400274, 'n_estimators(best_iteration+1)': 291}
CNN-BiLSTM-Attn: val RMSE 10.692
LSTM-Attn: val RMSE 10.853
RAW: checkpoint saved -> /content/beijing_Nongzhanguan_NO2/ckpt_h1_RAW.pkl

================ STL =======

In [ ]:

if 6 in HORIZONS:
    run_horizon(6)
    zip_and_download("checkpoint")
else:
    print("h = 6 skipped (not in HORIZONS)")


h = 6 skipped (not in HORIZONS)


In [1]:

if 24 in HORIZONS:
    run_horizon(24)
    zip_and_download("checkpoint")
else:
    print("h = 24 skipped (not in HORIZONS)")



################ HORIZON h = 24 ################
  set  n_samples        first_target         last_target  months_covered  lookback_h  horizon_h  decomp_window_h
train      23653 2013-03-16 22:00:00 2015-12-18 15:00:00              12          24         24              336
  val       3472 2015-12-18 18:00:00 2016-05-12 18:00:00               6          24         24              336
 test       6867 2016-05-12 19:00:00 2017-02-28 23:00:00              10          24         24              336

================ RAW ================
XGB best: {'max_depth': 3, 'learning_rate': 0.0860435499439301, 'subsample': 0.8214317602631983, 'colsample_bytree': 0.7796035803727068, 'min_child_weight': 8.0, 'gamma': 0.6079001961252244, 'reg_alpha': 0.0013282240456496195, 'reg_lambda': 0.20424534522558235, 'n_estimators(best_iteration+1)': 37}
CNN-BiLSTM-Attn: val RMSE 37.626
LSTM-Attn: val RMSE 37.524
RAW: checkpoint saved -> /content/beijing_Nongzhanguan_NO2/ckpt_h24_RAW.pkl

================ STL =

In [ ]:

SUMMARY_TEST, SUMMARY_DM, SUMMARY_SEASON = [], [], []
for HORIZON in HORIZONS:
    xlsx = f"{OUT_DIR}/{POLLUTANT}_h{HORIZON}_beijing_Nongzhanguan_NO2_results.xlsx"
    if not os.path.exists(xlsx):
        print(f"h = {HORIZON}: not finished yet - run its cell first"); continue
    _x = pd.ExcelFile(xlsx)
    for _sheet, _store in [("Table_test", SUMMARY_TEST), ("DM_tests_test", SUMMARY_DM),
                           ("Metrics_test_by_season", SUMMARY_SEASON)]:
        if _sheet in _x.sheet_names:
            _store.append(pd.read_excel(_x, _sheet).assign(h=HORIZON))
T = pd.concat(SUMMARY_TEST, ignore_index=True)
naive = T[T.Model == "Naive persistence"][["h", "RMSE", "MAE"]].rename(columns={"RMSE": "RMSE_naive", "MAE": "MAE_naive"})
T = T.merge(naive, on="h")
T["RMSE_skill_vs_persistence_%"] = 100 * (1 - T.RMSE / T.RMSE_naive)
T["MAE_skill_vs_persistence_%"] = 100 * (1 - T.MAE / T.MAE_naive)
wide = T.pivot_table(index=["Track", "Model"], columns="h", values=["MAE", "RMSE", "MASE", "R2"])
wide.columns = [f"{m}_h{h}" for m, h in wide.columns]
wide = wide.reset_index()
D = pd.concat(SUMMARY_DM, ignore_index=True) if SUMMARY_DM else pd.DataFrame()
summary_xlsx = f"{OUT_DIR}/{POLLUTANT}_multi_horizon_summary.xlsx"
with pd.ExcelWriter(summary_xlsx) as w:
    wide.to_excel(w, sheet_name="Test_metrics_by_horizon", index=False)
    T.to_excel(w, sheet_name="Test_metrics_long", index=False)
    if len(D):
        D[D.Reference == "Naive persistence"].to_excel(w, sheet_name="DM_vs_persistence", index=False)
        D[D.Reference.str.startswith("RAW")].to_excel(w, sheet_name="DM_decomposition_vs_RAW", index=False)
    if SUMMARY_SEASON:
        pd.concat(SUMMARY_SEASON, ignore_index=True).to_excel(w, sheet_name="Test_by_season", index=False)

fig, ax = plt.subplots(figsize=(8, 4.5))
for (track, model), g in T[T.Model.isin(["XGBoost", "Ensemble_Weighted"]) | (T.Track == "BASELINE")].groupby(["Track", "Model"]):
    g = g.sort_values("h")
    ax.plot(g.h, g.RMSE, marker="o", ls="--" if track == "BASELINE" else "-", label=f"{track} {model}")
ax.set_xlabel("Forecast horizon h (hours)"); ax.set_ylabel("Test RMSE (µg/m³)")
ax.set_xticks(sorted(T.h.unique())); ax.legend(fontsize=7, ncol=2); ax.set_title(f"{POLLUTANT}: test RMSE by horizon")
fig.tight_layout(); fig.savefig(f"{OUT_DIR}/rmse_by_horizon_{POLLUTANT}.png", dpi=300); plt.close(fig)

print("\n================ MULTI-HORIZON TEST SUMMARY ================")
show = T[T.Model.isin(["Naive persistence", "XGBoost", "Ensemble_Weighted", "Ensemble_Equal"])]
print(show[["h", "Track", "Model", "MAE", "RMSE", "MASE", "R2", "RMSE_skill_vs_persistence_%"]]
      .sort_values(["h", "RMSE"]).to_string(index=False, float_format="%.3f"))
if len(D):
    print("\nDecomposition vs RAW (DM, squared loss, test):")
    print(D[(D.Reference.str.startswith("RAW")) & (D.Loss == "squared")][["h", "Track", "Model", "DM_stat", "p_value", "better"]]
          .to_string(index=False, float_format="%.4f"))
print("\nSaved:", summary_xlsx)
print("All outputs in:", OUT_DIR)

zip_and_download("FINAL_results")
